In [1]:
from pathlib import Path
import time

import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

SEED = 42

In [2]:
PROJECT_ROOT = Path(r"C:\Case Study Trees\HomeCreditStudy")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULT_DIR = PROJECT_ROOT / "results"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


y_train = pd.read_csv(
    PROCESSED_DIR / "engineered" / "y_train.csv"
)["TARGET"]

y_valid = pd.read_csv(
    PROCESSED_DIR / "engineered" / "y_valid.csv"
)["TARGET"]

print("Training labels:", len(y_train))
print("Validation labels:", len(y_valid))

Training labels: 215256
Validation labels: 46127


In [3]:
X_train = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)

Training features: (215256, 320)
Validation features: (46127, 320)


In [4]:
rf_engineered = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

rf_engineered.fit(X_train, y_train)

rf_engineered_fit_seconds = time.perf_counter() - start_time

print(
    "Training time:",
    round(rf_engineered_fit_seconds, 2),
    "seconds"
)

Training time: 35.71 seconds


In [5]:
start_time = time.perf_counter()

valid_probability = rf_engineered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

rf_engineered_results = {
    "Model": "Random Forest",
    "Feature_Set": "Engineered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": rf_engineered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([rf_engineered_results]).round(4))

joblib.dump(
    rf_engineered,
    ARTIFACT_DIR / "rf_engineered_baseline.joblib"
)

pd.DataFrame([rf_engineered_results]).to_csv(
    RESULT_DIR / "rf_engineered_baseline_validation.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,Random Forest,Engineered,0.7429,0.2342,0.2541,35.7073,0.4062


In [6]:
X_train = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

rf_filtered = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

rf_filtered.fit(X_train, y_train)

rf_filtered_fit_seconds = time.perf_counter() - start_time

print("Training features:", X_train.shape)
print("Training time:", round(rf_filtered_fit_seconds, 2), "seconds")

Training features: (215256, 264)
Training time: 34.75 seconds


In [ ]:
start_time = time.perf_counter()

valid_probability = rf_filtered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

rf_filtered_results = {
    "Model": "Random Forest",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": rf_filtered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

rf_comparison = pd.DataFrame([
    rf_engineered_results,
    rf_filtered_results
]).sort_values("ROC_AUC", ascending=False)

display(rf_comparison.round(4))


joblib.dump(
    rf_filtered,
    ARTIFACT_DIR / "rf_filtered_baseline.joblib"
)

rf_comparison.to_csv(
    RESULT_DIR / "rf_baseline_comparison.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
1,Random Forest,Engineered + filtered,0.7464,0.2358,0.2527,34.7467,0.3759
0,Random Forest,Engineered,0.7429,0.2342,0.2541,35.7073,0.4062


In [9]:
import optuna

X_train = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_valid.csv",
    dtype="float32"
)

STUDY_DIR = PROJECT_ROOT / "studies"
STUDY_DIR.mkdir(parents=True, exist_ok=True)


def objective(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 100, 500, step=100
        ),
        max_depth=trial.suggest_int("max_depth", 8, 30),
        min_samples_split=trial.suggest_int(
            "min_samples_split", 2, 20
        ),
        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf", 1, 20
        ),
        max_features=trial.suggest_categorical(
            "max_features", ["sqrt", 0.3, 0.6]
        ),
        class_weight=trial.suggest_categorical(
            "class_weight", [None, "balanced"]
        ),
        random_state=SEED,
        n_jobs=4
    )

    start = time.perf_counter()

    model.fit(X_train, y_train)
    probability = model.predict_proba(X_valid)[:, 1]

    trial.set_user_attr(
        "fit_predict_seconds",
        time.perf_counter() - start
    )

    return roc_auc_score(y_valid, probability)


database_path = (STUDY_DIR / "home_credit_optuna.db").resolve()

study = optuna.create_study(
    study_name="rf_engineered_filtered_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

tuning_start = time.perf_counter()

study.optimize(
    objective,
    timeout=30 * 60,
    n_jobs=1,
    gc_after_trial=True
)

rf_tuning_seconds = time.perf_counter() - tuning_start

study.trials_dataframe().to_csv(
    RESULT_DIR / "rf_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", study.best_value)
print("Best parameters:", study.best_params)
print("Tuning seconds:", round(rf_tuning_seconds, 2))

c:\Users\gdeep\anaconda3\envs\100DaysML_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-10-16 00:15:03,798] A new study created in RDB with name: rf_engineered_filtered_v1
[I 2026-10-16 00:16:26,219] Trial 0 finished with value: 0.7514563028835408 and parameters: {'n_estimators': 200, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 0 with value: 0.7514563028835408.
[I 2026-10-16 00:17:29,267] Trial 1 finished with value: 0.7360237023437811 and parameters: {'n_estimators': 400, 'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 17, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7514563028835408.
[I 2026-10-16 00:30:28,730] Trial 2 finished with value: 0.743206469872364 and paramet

Best validation ROC-AUC: 0.7514563028835408
Best parameters: {'n_estimators': 200, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'class_weight': None}
Tuning seconds: 1991.99


In [10]:
print("Best trial:", study.best_trial.number)
print("Best validation ROC-AUC:", round(study.best_value, 4))

print("\nBest parameters:")
for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")

print("\nFiltered baseline ROC-AUC:")
print(round(rf_filtered_results["ROC_AUC"], 4))

improvement = study.best_value - rf_filtered_results["ROC_AUC"]
print("\nROC-AUC improvement:", round(improvement, 4))

Best trial: 0
Best validation ROC-AUC: 0.7515

Best parameters:
n_estimators: 200
max_depth: 29
min_samples_split: 15
min_samples_leaf: 12
max_features: sqrt
class_weight: None

Filtered baseline ROC-AUC:
0.7464

ROC-AUC improvement: 0.0051


In [11]:
rf_tuned = RandomForestClassifier(
    **study.best_params,
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

rf_tuned.fit(X_train, y_train)

rf_tuned_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = rf_tuned.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

rf_tuned_results = {
    "Model": "Random Forest",
    "Version": "Tuned",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": rf_tuned_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([rf_tuned_results]).round(4))

joblib.dump(
    rf_tuned,
    ARTIFACT_DIR / "rf_filtered_tuned.joblib"
)

pd.DataFrame([rf_tuned_results]).to_csv(
    RESULT_DIR / "rf_tuned_validation.csv",
    index=False
)

pd.Series(study.best_params, name="Value").to_csv(
    RESULT_DIR / "rf_best_parameters.csv",
    index_label="Parameter"
)

,Model,Version,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,Random Forest,Tuned,Engineered + filtered,0.7515,0.2402,0.2506,76.8446,1.1124


## XGBOOST

In [13]:
from xgboost import XGBClassifier

X_train = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

Training: (215256, 320)
Validation: (46127, 320)


In [14]:
xgb_engineered = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

xgb_engineered.fit(X_train, y_train)

xgb_engineered_fit_seconds = time.perf_counter() - start_time

print(
    "Training time:",
    round(xgb_engineered_fit_seconds, 2),
    "seconds"
)

Training time: 19.87 seconds


In [15]:
start_time = time.perf_counter()

valid_probability = xgb_engineered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

xgb_engineered_results = {
    "Model": "XGBoost",
    "Feature_Set": "Engineered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": xgb_engineered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([xgb_engineered_results]).round(4))

joblib.dump(
    xgb_engineered,
    ARTIFACT_DIR / "xgb_engineered_baseline.joblib"
)

pd.DataFrame([xgb_engineered_results]).to_csv(
    RESULT_DIR / "xgb_engineered_baseline_validation.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,XGBoost,Engineered,0.7607,0.2536,0.245,19.8692,0.1865


In [16]:
X_train = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

# Copy the engineered baseline settings, without copying fitted trees
xgb_filtered = XGBClassifier(**xgb_engineered.get_params())

start_time = time.perf_counter()

xgb_filtered.fit(X_train, y_train)

xgb_filtered_fit_seconds = time.perf_counter() - start_time

print("Training features:", X_train.shape)
print(
    "Training time:",
    round(xgb_filtered_fit_seconds, 2),
    "seconds"
)

Training features: (215256, 264)
Training time: 24.11 seconds


In [17]:
start_time = time.perf_counter()

valid_probability = xgb_filtered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

xgb_filtered_results = {
    "Model": "XGBoost",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": xgb_filtered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

xgb_comparison = pd.DataFrame([
    xgb_engineered_results,
    xgb_filtered_results
]).sort_values("ROC_AUC", ascending=False)

display(xgb_comparison.round(4))

joblib.dump(
    xgb_filtered,
    ARTIFACT_DIR / "xgb_filtered_baseline.joblib"
)

xgb_comparison.to_csv(
    RESULT_DIR / "xgb_baseline_comparison.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
1,XGBoost,Engineered + filtered,0.7616,0.2530,0.2449,24.1122,0.2012
0,XGBoost,Engineered,0.7607,0.2536,0.2450,19.8692,0.1865


In [21]:
def xgb_objective(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        max_depth=trial.suggest_int("max_depth", 3, 9),
        min_child_weight=trial.suggest_float(
            "min_child_weight", 1, 30, log=True
        ),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        reg_alpha=trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        reg_lambda=trial.suggest_float(
            "reg_lambda", 0.1, 20, log=True
        ),
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=SEED,
        n_jobs=4
    )

    start = time.perf_counter()

    model.fit(X_train, y_train)
    probability = model.predict_proba(X_valid)[:, 1]

    trial.set_user_attr(
        "fit_predict_seconds",
        time.perf_counter() - start
    )

    return roc_auc_score(y_valid, probability)


tuning_start = time.perf_counter()

xgb_study.optimize(
    xgb_objective,
    timeout=30 * 60,
    n_jobs=1,
    gc_after_trial=True
)

xgb_tuning_seconds = time.perf_counter() - tuning_start

xgb_study.trials_dataframe().to_csv(
    RESULT_DIR / "xgb_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", xgb_study.best_value)
print("Best parameters:", xgb_study.best_params)
print("Tuning seconds:", round(xgb_tuning_seconds, 2))

[I 2026-10-16 01:03:20,202] Trial 0 finished with value: 0.7437435141348574 and parameters: {'n_estimators': 500, 'learning_rate': 0.13125830316209655, 'max_depth': 8, 'min_child_weight': 7.6611007077713635, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 9.842315738502599}. Best is trial 0 with value: 0.7437435141348574.
[I 2026-10-16 01:03:48,570] Trial 1 finished with value: 0.7632560051825367 and parameters: {'n_estimators': 700, 'learning_rate': 0.06803900745073706, 'max_depth': 3, 'min_child_weight': 27.08160864249967, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.0008111941985431928, 'reg_lambda': 0.26425260575499165}. Best is trial 1 with value: 0.7632560051825367.
[I 2026-10-16 01:04:12,946] Trial 2 finished with value: 0.7645640769089128 and parameters: {'n_estimators': 400, 'learning_rate': 0.04141536110538596, 'max_depth': 6, 'min_child_weight': 2.69265525148

Best validation ROC-AUC: 0.7645760838416247
Best parameters: {'n_estimators': 500, 'learning_rate': 0.031210347442980457, 'max_depth': 6, 'min_child_weight': 2.06260932932417, 'subsample': 0.8355766793979985, 'colsample_bytree': 0.7044709053513115, 'reg_alpha': 0.0015054671981229908, 'reg_lambda': 0.4068042227014351}
Tuning seconds: 1829.0


In [22]:
xgb_tuned = XGBClassifier(
    **xgb_study.best_params,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

xgb_tuned.fit(X_train, y_train)

xgb_tuned_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = xgb_tuned.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

xgb_tuned_results = {
    "Model": "XGBoost",
    "Version": "Tuned",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": xgb_tuned_fit_seconds,
    "Prediction_Seconds": prediction_seconds,
    "Tuning_Seconds": xgb_tuning_seconds
}

display(pd.DataFrame([xgb_tuned_results]).round(4))

joblib.dump(
    xgb_tuned,
    ARTIFACT_DIR / "xgb_filtered_tuned.joblib"
)

pd.DataFrame([xgb_tuned_results]).to_csv(
    RESULT_DIR / "xgb_tuned_validation.csv",
    index=False
)

pd.Series(xgb_study.best_params, name="Value").to_csv(
    RESULT_DIR / "xgb_best_parameters.csv",
    index_label="Parameter"
)

,Model,Version,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,XGBoost,Tuned,Engineered + filtered,0.7646,0.2552,0.2441,30.498,0.1958,1829.0011


## LIGHT GBM

In [23]:
from lightgbm import LGBMClassifier

X_train = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

Training: (215256, 320)
Validation: (46127, 320)


In [25]:
assert X_train.columns.equals(X_valid.columns)

safe_names = [f"feature_{i}" for i in range(X_train.shape[1])]

feature_mapping = pd.DataFrame({
    "Original_Name": X_train.columns,
    "LightGBM_Name": safe_names
})

feature_mapping.to_csv(
    ARTIFACT_DIR / "lgb_engineered_feature_mapping.csv",
    index=False
)

X_train.columns = safe_names
X_valid.columns = safe_names

print("Feature names updated. Values unchanged.")

Feature names updated. Values unchanged.


In [26]:
lgb_engineered = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary",
    random_state=SEED,
    n_jobs=4,
    verbosity=-1
)

start_time = time.perf_counter()

lgb_engineered.fit(X_train, y_train)

lgb_engineered_fit_seconds = time.perf_counter() - start_time

print(
    "Training time:",
    round(lgb_engineered_fit_seconds, 2),
    "seconds"
)

Training time: 10.0 seconds


In [27]:
start_time = time.perf_counter()

valid_probability = lgb_engineered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

lgb_engineered_results = {
    "Model": "LightGBM",
    "Feature_Set": "Engineered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": lgb_engineered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([lgb_engineered_results]).round(4))

joblib.dump(
    lgb_engineered,
    ARTIFACT_DIR / "lgb_engineered_baseline.joblib"
)

pd.DataFrame([lgb_engineered_results]).to_csv(
    RESULT_DIR / "lgb_engineered_baseline_validation.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,LightGBM,Engineered,0.7609,0.2521,0.2452,10.0042,0.4699


In [ ]:
X_train = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

safe_names = [f"feature_{i}" for i in range(X_train.shape[1])]

filtered_feature_mapping = pd.DataFrame({
    "Original_Name": X_train.columns,
    "LightGBM_Name": safe_names
})

filtered_feature_mapping.to_csv(
    ARTIFACT_DIR / "lgb_filtered_feature_mapping.csv",
    index=False
)

X_train.columns = safe_names
X_valid.columns = safe_names

lgb_filtered = LGBMClassifier(**lgb_engineered.get_params())

start_time = time.perf_counter()

lgb_filtered.fit(X_train, y_train)

lgb_filtered_fit_seconds = time.perf_counter() - start_time

print("Training features:", X_train.shape)
print(
    "Training time:",
    round(lgb_filtered_fit_seconds, 2),
    "seconds"
)

Training features: (215256, 264)
Training time: 9.56 seconds


In [30]:
start_time = time.perf_counter()

valid_probability = lgb_filtered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

lgb_filtered_results = {
    "Model": "LightGBM",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": lgb_filtered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

lgb_comparison = pd.DataFrame([
    lgb_engineered_results,
    lgb_filtered_results
]).sort_values("ROC_AUC", ascending=False)

display(lgb_comparison.round(4))

joblib.dump(
    lgb_filtered,
    ARTIFACT_DIR / "lgb_filtered_baseline.joblib"
)

lgb_comparison.to_csv(
    RESULT_DIR / "lgb_baseline_comparison.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
1,LightGBM,Engineered + filtered,0.7614,0.2494,0.2452,9.5629,0.4408
0,LightGBM,Engineered,0.7609,0.2521,0.2452,10.0042,0.4699


In [31]:
def lgb_objective(trial):
    model = LGBMClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        num_leaves=trial.suggest_int("num_leaves", 15, 127),
        max_depth=-1,
        min_child_samples=trial.suggest_int(
            "min_child_samples", 20, 150
        ),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        reg_alpha=trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        reg_lambda=trial.suggest_float(
            "reg_lambda", 0.1, 20, log=True
        ),
        objective="binary",
        random_state=SEED,
        n_jobs=4,
        verbosity=-1
    )

    start = time.perf_counter()

    model.fit(X_train, y_train)
    probability = model.predict_proba(X_valid)[:, 1]

    trial.set_user_attr(
        "fit_predict_seconds",
        time.perf_counter() - start
    )

    return roc_auc_score(y_valid, probability)


lgb_study = optuna.create_study(
    study_name="lgb_engineered_filtered_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

tuning_start = time.perf_counter()

lgb_study.optimize(
    lgb_objective,
    timeout=30 * 60,
    n_jobs=1,
    gc_after_trial=True
)

lgb_tuning_seconds = time.perf_counter() - tuning_start

lgb_study.trials_dataframe().to_csv(
    RESULT_DIR / "lgb_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", lgb_study.best_value)
print("Best parameters:", lgb_study.best_params)
print("Tuning seconds:", round(lgb_tuning_seconds, 2))

[I 2026-10-16 02:06:19,508] A new study created in RDB with name: lgb_engineered_filtered_v1
[I 2026-10-16 02:06:36,854] Trial 0 finished with value: 0.7314378519769631 and parameters: {'n_estimators': 500, 'learning_rate': 0.13125830316209655, 'num_leaves': 97, 'min_child_samples': 98, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 9.842315738502599}. Best is trial 0 with value: 0.7314378519769631.
[I 2026-10-16 02:06:48,660] Trial 1 finished with value: 0.7594988389878683 and parameters: {'n_estimators': 700, 'learning_rate': 0.06803900745073706, 'num_leaves': 17, 'min_child_samples': 147, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.0008111941985431928, 'reg_lambda': 0.26425260575499165}. Best is trial 1 with value: 0.7594988389878683.
[I 2026-10-16 02:07:00,726] Trial 2 finished with value: 0.7625543310538823 and parameters: {'n_estimators': 400, 'learning_rate': 0

Best validation ROC-AUC: 0.7651776748665996
Best parameters: {'n_estimators': 600, 'learning_rate': 0.013533950488442989, 'num_leaves': 95, 'min_child_samples': 144, 'subsample': 0.6530740311316475, 'colsample_bytree': 0.7860198467382739, 'reg_alpha': 0.0008533477368853772, 'reg_lambda': 1.8738619464917554}
Tuning seconds: 1820.31


In [32]:
lgb_tuned = LGBMClassifier(
    **lgb_study.best_params,
    max_depth=-1,
    subsample_freq=1,
    objective="binary",
    random_state=SEED,
    n_jobs=4,
    verbosity=-1
)

start_time = time.perf_counter()

lgb_tuned.fit(X_train, y_train)

lgb_tuned_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = lgb_tuned.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

lgb_tuned_results = {
    "Model": "LightGBM",
    "Version": "Tuned",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": lgb_tuned_fit_seconds,
    "Prediction_Seconds": prediction_seconds,
    "Tuning_Seconds": lgb_tuning_seconds
}

display(pd.DataFrame([lgb_tuned_results]).round(4))

joblib.dump(
    lgb_tuned,
    ARTIFACT_DIR / "lgb_filtered_tuned.joblib"
)

pd.DataFrame([lgb_tuned_results]).to_csv(
    RESULT_DIR / "lgb_tuned_validation.csv",
    index=False
)

pd.Series(lgb_study.best_params, name="Value").to_csv(
    RESULT_DIR / "lgb_best_parameters.csv",
    index_label="Parameter"
)

,Model,Version,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,LightGBM,Tuned,Engineered + filtered,0.7652,0.2552,0.2441,21.3769,1.9395,1820.3074


## CATBOOST

In [33]:
from catboost import CatBoostClassifier

X_train = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

Training: (215256, 320)
Validation: (46127, 320)


In [34]:
cat_baseline_params = {
    "iterations": 300,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "thread_count": 4,
    "verbose": False,
    "allow_writing_files": False
}

cat_engineered = CatBoostClassifier(**cat_baseline_params)

start_time = time.perf_counter()

cat_engineered.fit(X_train, y_train)

cat_engineered_fit_seconds = time.perf_counter() - start_time

print("Training seconds:", round(cat_engineered_fit_seconds, 2))

Training seconds: 29.01


In [35]:
start_time = time.perf_counter()

valid_probability = cat_engineered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

cat_engineered_results = {
    "Model": "CatBoost",
    "Feature_Set": "Engineered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": cat_engineered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([cat_engineered_results]).round(4))

joblib.dump(
    cat_engineered,
    ARTIFACT_DIR / "cat_engineered_baseline.joblib"
)

pd.DataFrame([cat_engineered_results]).to_csv(
    RESULT_DIR / "cat_engineered_baseline_validation.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,CatBoost,Engineered,0.7619,0.2537,0.2449,29.0052,0.0504


In [36]:
X_train = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / "engineered_filtered" / "X_valid.csv",
    dtype="float32"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)

cat_filtered = CatBoostClassifier(**cat_baseline_params)

start_time = time.perf_counter()

cat_filtered.fit(X_train, y_train)

cat_filtered_fit_seconds = time.perf_counter() - start_time

print("Training seconds:", round(cat_filtered_fit_seconds, 2))

Training seconds: 24.5


In [37]:
start_time = time.perf_counter()

valid_probability = cat_filtered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

cat_filtered_results = {
    "Model": "CatBoost",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": cat_filtered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

cat_comparison = pd.DataFrame([
    cat_engineered_results,
    cat_filtered_results
]).sort_values("ROC_AUC", ascending=False)

display(cat_comparison.round(4))

joblib.dump(
    cat_filtered,
    ARTIFACT_DIR / "cat_filtered_baseline.joblib"
)

cat_comparison.to_csv(
    RESULT_DIR / "cat_baseline_comparison.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
1,CatBoost,Engineered + filtered,0.7628,0.2562,0.2446,24.4953,0.0441
0,CatBoost,Engineered,0.7619,0.2537,0.2449,29.0052,0.0504


In [38]:
if cat_filtered_results["ROC_AUC"] > cat_engineered_results["ROC_AUC"]:
    cat_feature_folder = "engineered_filtered"
    cat_feature_label = "Engineered + filtered"
    cat_selected_baseline = cat_filtered_results
else:
    cat_feature_folder = "engineered"
    cat_feature_label = "Engineered"
    cat_selected_baseline = cat_engineered_results

X_train = pd.read_csv(
    PROCESSED_DIR / cat_feature_folder / "X_train.csv",
    dtype="float32"
)

X_valid = pd.read_csv(
    PROCESSED_DIR / cat_feature_folder / "X_valid.csv",
    dtype="float32"
)

print("Selected:", cat_feature_label)
print("Baseline validation ROC-AUC:", cat_selected_baseline["ROC_AUC"])

Selected: Engineered + filtered
Baseline validation ROC-AUC: 0.7627637494388215


In [40]:
def cat_objective(trial):
    model = CatBoostClassifier(
        iterations=trial.suggest_int(
            "iterations", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        depth=trial.suggest_int("depth", 4, 8),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg", 1, 30, log=True
        ),
        random_strength=trial.suggest_float(
            "random_strength", 0.01, 10, log=True
        ),
        bootstrap_type="Bayesian",
        bagging_temperature=trial.suggest_float(
            "bagging_temperature", 0, 5
        ),
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        thread_count=4,
        verbose=False,
        allow_writing_files=False
    )

    start = time.perf_counter()

    model.fit(X_train, y_train)
    probability = model.predict_proba(X_valid)[:, 1]

    trial.set_user_attr(
        "fit_predict_seconds",
        time.perf_counter() - start
    )

    return roc_auc_score(y_valid, probability)

In [41]:
cat_study = optuna.create_study(
    study_name=f"cat_onehot_{cat_feature_folder}_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

tuning_start = time.perf_counter()

cat_study.optimize(
    cat_objective,
    timeout=30 * 60,
    n_jobs=1,
    gc_after_trial=True
)

cat_tuning_seconds = time.perf_counter() - tuning_start

cat_study.trials_dataframe().to_csv(
    RESULT_DIR / "cat_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", cat_study.best_value)
print("Best parameters:", cat_study.best_params)
print("Tuning seconds:", round(cat_tuning_seconds, 2))

[I 2026-10-16 02:41:30,831] Using an existing study with `study_name='cat_onehot_engineered_filtered_v1'` instead of creating a new one.
[I 2026-10-16 02:42:16,771] Trial 0 finished with value: 0.762559213619874 and parameters: {'iterations': 500, 'learning_rate': 0.13125830316209655, 'depth': 7, 'l2_leaf_reg': 7.6611007077713635, 'random_strength': 0.02938027938703535, 'bagging_temperature': 0.7799726016810132}. Best is trial 0 with value: 0.762559213619874.
[I 2026-10-16 02:42:36,015] Trial 1 finished with value: 0.7525799833336682 and parameters: {'iterations': 200, 'learning_rate': 0.10440040750544663, 'depth': 7, 'l2_leaf_reg': 11.114989443094977, 'random_strength': 0.011527987128232402, 'bagging_temperature': 4.8495492608099715}. Best is trial 0 with value: 0.762559213619874.
[I 2026-10-16 02:43:31,775] Trial 2 finished with value: 0.7545914485358672 and parameters: {'iterations': 900, 'learning_rate': 0.01777174904859463, 'depth': 4, 'l2_leaf_reg': 1.8659959624904914, 'random_st

Best validation ROC-AUC: 0.7657613029882849
Best parameters: {'iterations': 700, 'learning_rate': 0.051367912293047865, 'depth': 6, 'l2_leaf_reg': 2.53181575966548, 'random_strength': 0.4861048354419499, 'bagging_temperature': 0.20104346523931832}
Tuning seconds: 1844.64


In [42]:
cat_tuned = CatBoostClassifier(
    **cat_study.best_params,
    bootstrap_type="Bayesian",
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    thread_count=4,
    verbose=False,
    allow_writing_files=False
)

start_time = time.perf_counter()

cat_tuned.fit(X_train, y_train)

cat_tuned_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = cat_tuned.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

cat_tuned_results = {
    "Model": "CatBoost",
    "Version": "Tuned",
    "Feature_Set": cat_feature_label,
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": cat_tuned_fit_seconds,
    "Prediction_Seconds": prediction_seconds,
    "Tuning_Seconds": cat_tuning_seconds
}

display(pd.DataFrame([cat_tuned_results]).round(4))

joblib.dump(
    cat_tuned,
    ARTIFACT_DIR / f"cat_{cat_feature_folder}_tuned.joblib"
)

pd.DataFrame([cat_tuned_results]).to_csv(
    RESULT_DIR / "cat_tuned_validation.csv",
    index=False
)

pd.Series(cat_study.best_params, name="Value").to_csv(
    RESULT_DIR / "cat_best_parameters.csv",
    index_label="Parameter"
)

,Model,Version,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,CatBoost,Tuned,Engineered + filtered,0.7658,0.2563,0.2437,53.1228,0.0369,1844.6371
